In [1]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime
from pathlib import Path
import json, os, re

# ── Ollama ──
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL    = "qwen3.8"          # or mistral, qwen2, phi3 …
OLLAMA_TIMEOUT  = 180

_cwd = Path.cwd()
# If launched from the parent folder, go into NepalNewsAgent/
# If launched from inside NepalNewsAgent/, stay here
OUTPUT_DIR = _cwd / "NepalNewsAgent" if (_cwd / "NepalNewsAgent").is_dir() else _cwd
OUTPUT_FILE = str(OUTPUT_DIR / "nepali_news.html")

print(f"📁 Output dir: {OUTPUT_DIR.resolve()}")

# ── Devanagari Unicode range (U+0900 – U+097F) ──
DEVANAGARI_RE = re.compile(r'[\u0900-\u097F]')

def has_devanagari(text: str) -> bool:
    return bool(DEVANAGARI_RE.search(text))

print(f"✅ Ollama : {OLLAMA_BASE_URL}  |  Model: {OLLAMA_MODEL}")



📁 Output dir: C:\Users\kusha\AgenticAIWs\NepalNewsAgent
✅ Ollama : http://localhost:11434  |  Model: qwen3.8


In [2]:
#Verify Ollama is Alive
def check_ollama():
    try:
        r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        r.raise_for_status()
        tags = r.json().get("models", [])
        print(f"🟢 Ollama is running. Available models: {len(tags)}")
        for m in tags:
            print(f"   • {m['name']}")
        return True
    except requests.exceptions.ConnectionError:
        print("🔴 Ollama is NOT running. Start it with: ollama serve")
        return False

check_ollama()


🟢 Ollama is running. Available models: 5
   • qwen3.8:latest
   • gpt-oss:20b
   • qwen2.5-coder:1.5b-base
   • nomic-embed-text:latest
   • llama3.1:8b


True

In [3]:
import json, re, time

def _raw_ollama(model: str, prompt: str, system: str = "",
                temperature: float = 0.5, num_predict: int = 2048) -> dict:
    """Low-level call — returns the FULL JSON so we can debug."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_predict": num_predict,
            "num_ctx": 4096,          # ← important: enough context window
        },
    }
    r = requests.post(f"{OLLAMA_BASE_URL}/api/chat",
                      json=payload, timeout=OLLAMA_TIMEOUT)
    r.raise_for_status()
    data = r.json()

    # Ollama sometimes nests the text differently
    content = ""
    if "message" in data:
        content = data["message"].get("content", "")
    elif "response" in data:
        content = data["response"]
    elif "choices" in data:
        content = data["choices"][0].get("message", {}).get("content", "")

    return {"content": content.strip(), "raw": data}


def ollama_with_retry(model: str, prompt: str, system: str = "",
                      temperature: float = 0.5,
                      max_retries: int = 3) -> str:
    """Try up to `max_retries` times; return first non-empty answer."""
    last_raw = {}
    for attempt in range(1, max_retries + 1):
        print(f"    … attempt {attempt}/{max_retries}", end=" ", flush=True)
        try:
            result = _raw_ollama(model, prompt, system,
                                 temperature=temperature)
            content = result["content"]
            last_raw = result["raw"]

            if content and len(content) > 20:
                print(f"→ OK ({len(content)} chars)")
                return content

            # ── debug dump when empty ──
            print(f"→ EMPTY")
            print(f"      raw keys : {list(result['raw'].keys())}")
            print(f"      raw dump : {json.dumps(result['raw'], ensure_ascii=False)[:600]}")
            time.sleep(2 * attempt)

        except requests.exceptions.Timeout:
            print("→ TIMEOUT, retrying …")
            time.sleep(3)
        except Exception as e:
            print(f"→ ERROR {e}")
            last_raw = {"error": str(e)}
            time.sleep(2)

    print(f"\n    ⚠️  All {max_retries} attempts failed.")
    print(f"    Last raw: {json.dumps(last_raw, ensure_ascii=False)[:500]}")
    return ""


In [4]:
#Multi Site Scraper
import requests, warnings, urllib3
from bs4 import BeautifulSoup
from datetime import datetime
import json, os, re

# ── Silence the "InsecureRequestWarning" we intentionally trigger ──
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/125.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "ne,nq;q=0.9,en;q=0.5",
}

# ═══════════════════════════════════════════════════
#  safe_get  –  HTTPS → HTTP → verify=False cascade
# ═══════════════════════════════════════════════════
def safe_get(url: str, timeout: int = 20):
    """
    Try three strategies in order:
      1.  HTTPS  (normal)
      2.  HTTP   (plain)
      3.  HTTPS  (verify=False)   ← fixes cert-mismatch sites
    Returns (response, strategy_used) or (None, error_string).
    """
    strategies = [
        ("https normal",    {"verify": True,  "url": url}),
        ("http fallback",   {"verify": True,  "url": url.replace("https://", "http://")}),
        ("https no-verify", {"verify": False, "url": url}),
    ]
    last_err = ""
    for label, kw in strategies:
        u = kw["url"]
        try:
            r = requests.get(u, headers=HEADERS,
                             timeout=timeout, verify=kw["verify"])
            r.raise_for_status()
            return r, label
        except requests.exceptions.SSLError as e:
            last_err = f"[SSL] {e.__class__.__name__}"
        except requests.exceptions.ConnectionError as e:
            msg = str(e)
            if "getaddrinfo" in msg:
                last_err = f"[DNS] host not found"
            elif "Max retries" in msg:
                last_err = f"[CONN] {msg[:80]}"
            else:
                last_err = f"[CONN] {e.__class__.__name__}"
        except requests.exceptions.Timeout:
            last_err = "[TIMEOUT]"
        except Exception as e:
            last_err = f"[{e.__class__.__name__}] {str(e)[:60]}"
        # small back-off before next strategy
        import time; time.sleep(1)
    return None, last_err


# ═══════════════════════════════════════════════════
#  Sources  –  Devanagari-Nepali only
#  (added backup domains where the primary may be down)
# ═══════════════════════════════════════════════════
SOURCES = [
    # (display_name,  [primary_url,  backup_url,  …])
    ("ऑनलाइन खबर", ["https://onlinekhabar.com/",
                       "http://onlinekhabar.com/"]),
    ("Nepal News",    ["https://nepalnews.com/",
                       "http://nepalnews.com/"]),
    ("ई-काठमाडौं",    ["https://ekantipur.com/",
                       "http://ekantipur.com/"]),
    ("सेतोपाटी",       ["https://setopati.com/",
                       "http://setopati.com/"]),
    ("गोरखापत्र",       ["https://gorkhapatraonline.com/",
                       "http://gorkhapatraonline.com/"]),
    ("Nagarik News", ["https://nagariknews.nagariknetwork.com/",
                       "http://nagariknews.nagariknetwork.com/"]),
    ("भैरवी",          ["https://bhairabipost.com/",
                       "http://bhairabipost.com/"]),
    ("काठमाडौं पोस्ट",  ["https://kathmandupost.com/",
                       "http://kathmandupost.com/"]),
]

# ═══════════════════════════════════════════════════
#  Article-title selectors  (WordPress + custom)
# ═══════════════════════════════════════════════════
ARTICLE_SELECTOR = (
    "article, "
    "h2.article-title, h3.article-title, "
    "h2.post-title,   h3.post-title, "
    "h2.entry-title,  h3.entry-title, "
    "h2.news-title,   h3.news-title, "
    "h2.story-title,  h3.story-title, "
    "h2.headline,     h3.headline, "
    ".article-title,  .post-title, "
    ".entry-title,    .news-title, "
    ".story-title,    .headline, "
    ".news-list .item-title, "
    ".featured-articles .title, "
    ".top-news .title, "
    ".home-news a, "
    ".latest-news .title"
)

EXCLUDE_SELECTORS = (
    "nav, header, footer, aside, "
    ".navbar, .menu, .sidebar, .widget, "
    ".category, .tag, .breadcrumb, "
    ".pagination, .social, .share, "
    ".comment, .comments, "
    ".author, .profile, "
    ".footer, .header, .banner, "
    ".ad, .advert, .ads, "
    ".related-posts, .related, "
    ".trending-sidebar, .most-read"
)

DEVANAGARI_RE = re.compile(r'[\u0900-\u097F]')
MIN_TITLE_LEN = 30
MAX_PER_SITE  = 12

BAD_WORDS = {
    "लोगिन","साइन इन","सुची","श्रेणी","ट्याग",
    "टिप्पणी","साझा","फ्यासबुक","ट्विटर",
    "कुनै पनि","पढ्नुहोस्","थप","विस्तृत",
    "All","Home","Contact","About",
}


def has_devanagari(t): return bool(DEVANAGARI_RE.search(t))


def _is_valid_title(text: str) -> bool:
    t = text.strip()
    if len(t) < MIN_TITLE_LEN: return False
    if not has_devanagari(t):  return False
    return not any(w in t for w in BAD_WORDS)


def _base(url): 
    p = url.rsplit("/", 1)
    return p[0] if len(p) > 1 else url


def scrape_one(name: str, urls: list):
    """Try each URL in the list until one works."""
    for url in urls:
        print(f"  📰 {name:.<28s} {url[:45]}", end=" ", flush=True)
        resp, strategy = safe_get(url)
        if resp is None:
            print(f"→ ✗ {strategy}")
            continue                       # try next backup URL

        soup = BeautifulSoup(resp.text, "lxml")
        base = _base(resp.url)

        # strip nav / sidebar / footer
        for sel in [s.strip() for s in EXCLUDE_SELECTORS.split(",")]:
            for el in soup.select(sel):
                el.decompose()

        # collect candidate anchors
        candidates = []
        for anchor in soup.select(
            f"{ARTICLE_SELECTOR} a, a {ARTICLE_SELECTOR}"
        ):
            href  = anchor.get("href", "")
            title = anchor.get_text(strip=True)
            if _is_valid_title(title) and href:
                full = href if href.startswith("http") else f"{base}{href}"
                candidates.append({"source": name, "title": title, "url": full})

        # fallback: h2 / h3 headings
        if len(candidates) < 5:
            for h in soup.select("h2, h3"):
                title = h.get_text(strip=True)
                a = h.find("a")
                href  = a.get("href", "") if a else ""   # ← FIXED: .get() instead of []
                if _is_valid_title(title) and href:
                    full = href if href.startswith("http") else f"{base}{href}"
                    e = {"source": name, "title": title, "url": full}
                    if e not in candidates:
                        candidates.append(e)

        # dedup
        seen, unique = set(), []
        for c in candidates:
            if c["url"] not in seen:
                seen.add(c["url"])
                unique.append(c)
            if len(unique) >= MAX_PER_SITE:
                break

        print(f"→ ✓ {len(unique)}  [{strategy}]")
        return unique

    print(f"  📰 {name:.<28s} → ✗ all URLs failed")
    return []


# ── run the scraper ──
all_articles = []
for name, urls in SOURCES:
    all_articles.extend(scrape_one(name, urls))

# global dedup
seen, unique = set(), []
for a in all_articles:
    if a["url"] not in seen and has_devanagari(a["title"]):
        seen.add(a["url"])
        unique.append(a)

TARGET = 50
top_articles = unique[:TARGET]

print(f"\n{'='*58}")
print(f"  Total : {len(unique)}   |   Picked : {len(top_articles)}")
print(f"{'='*58}\n")
for i, a in enumerate(top_articles, 1):
    print(f"  {i:>2}. [{a['source']}] {a['title'][:65]}")

assert len(top_articles) >= 10, "Too few articles – check connectivity"


  📰 ऑनलाइन खबर.................. https://onlinekhabar.com/ → ✓ 12  [https normal]
  📰 Nepal News.................. https://nepalnews.com/ → ✓ 2  [https normal]
  📰 ई-काठमाडौं.................. https://ekantipur.com/ → ✓ 12  [https normal]
  📰 सेतोपाटी.................... https://setopati.com/ → ✓ 0  [http fallback]
  📰 गोरखापत्र................... https://gorkhapatraonline.com/ → ✓ 12  [https normal]
  📰 Nagarik News................ https://nagariknews.nagariknetwork.com/ → ✓ 0  [https normal]
  📰 भैरवी....................... https://bhairabipost.com/ → ✗ [DNS] host not found
  📰 भैरवी....................... http://bhairabipost.com/ → ✗ [DNS] host not found
  📰 भैरवी....................... → ✗ all URLs failed
  📰 काठमाडौं पोस्ट.............. https://kathmandupost.com/ → ✓ 0  [https normal]

  Total : 38   |   Picked : 38

   1. [ऑनलाइन खबर] एआईले बदल्दैछ अपराधको शैली : साइबर हमलादेखि सूचना हेरफेरसम्म
   2. [ऑनलाइन खबर] वीरमा दुर्लभ शल्यक्रिया : मस्तिष्कमा फुलेका ६ रक्तनली एकैपटक बन्द
 

In [5]:
# ── Build the headline list ──
digest_lines = [f"{i}. {a['title']}" for i, a in enumerate(top_articles, 1)]
digest_text  = "\n".join(digest_lines)
print(f"  (sending {len(digest_lines)} headlines to {OLLAMA_MODEL})\n")

ai_summary = ""

# ── TIER 1: Devanagari system prompt (ideal case) ──
print("  [Tier 1] Devanagari system prompt …")
ai_summary = ollama_with_retry(
    model=OLLAMA_MODEL,
    system="तपाईं एक नेपाली समाचार विश्लेषक हुनुहुन्छ। देवनागरी नेपालीमा जवाफ दिनुहोस्।",
    prompt=(
        "तल आजका शीर्ष नेपाली समाचार शीर्षकहरू छन्। "
        "३-४ वाक्यको सारांश देवनागरी नेपालीमा लेख्नुहोस्।\n\n"
        + digest_text
    ),
    temperature=0.5,
)

# ── TIER 2: No system prompt, single prompt ──
if not ai_summary:
    print("  [Tier 2] Single-prompt fallback …")
    ai_summary = ollama_with_retry(
        model=OLLAMA_MODEL,
        prompt=(
            "You are a Nepali news analyst. "
            "Read the following headlines and write a 3-4 sentence summary "
            "IN DEVANAGARI NEPALI ONLY (देवनागरी नेपाली).\n\n"
            + digest_text
        ),
        temperature=0.6,
        max_retries=2,
    )

# ── TIER 3: Plain English summary (last resort) ──
if not ai_summary:
    print("  [Tier 3] English fallback …")
    ai_summary = ollama_with_retry(
        model=OLLAMA_MODEL,
        prompt=(
            "Summarize these Nepali news headlines in 3 sentences.\n\n"
            + digest_text
        ),
        temperature=0.6,
        max_retries=2,
    )

# ── Final guard ──
if not ai_summary:
    ai_summary = ("सारांश उपलब्ध छैन। कृपया Ollama चालु छ "
                  "र मोडेल (`ollama list`) install छ भन्ने कुरा पक्का गर्नुहोस्।")

print(f"\n✅ AI सारांश:\n{ai_summary}\n")


  (sending 38 headlines to qwen3.8)

  [Tier 1] Devanagari system prompt …
    … attempt 1/3 → OK (629 chars)

✅ AI सारांश:
आजका शीर्ष समाचारमा नेपाल–चीन व्यापारको 'लाइफलाइन' मानिने रसुवागढी नाकाको अवस्था, ग्यास अभावले उपभोक्ताको मार र गल्छी–रसुवागढी सडकको ट्र्याक खोल्ने प्रगति सबैभन्दा प्रमुख विषय बनेको छ। राजनीतिक क्षेत्रमा 'देउवाको ग्रेसफुल एक्जिट' र नयाँ पुस्ताको नेतृत्वको माग उठेको छ भने एआईले अपराधको शैली बदल्दै गरेको, साइबर हमलादेखि सूचना हेरफेरसम्मको चर्चा पनि छ। स्वास्थ्य, श्रमिकको मृत्युपछि शव फर्काउने संकट, भोटेकोशी बाढीपछि कुलेखानीको दबाब र सर्वोच्चमा मुद्दा फर्छ्योटको गति बढ्ने जस्ता विषय पनि आजको समाचारमा उल्लेखनीय छन्। यसबाहेक नेपालले एसीसी प्रिमियर कप जित्दै एसिया कपमा छनोट हुने खेलाडीगत सफलता पनि आजको प्रमुख समाचारमध्ये एक हो।



In [6]:
#Generate Hyler linked html page
def build_html(articles, ai_summary="", output=OUTPUT_FILE):
    today = datetime.now().strftime("%Y/%m/%d")

    grouped = {}
    for a in articles:
        grouped.setdefault(a["source"], []).append(a)

    rows_html = ""
    num = 0
    for source, items in grouped.items():
        rows_html += f'\n    <div class="source-block">'
        rows_html += f'<h2>📰 {source}</h2>\n    <ol start="{num+1}">'
        for a in items:
            num += 1
            # escape HTML entities in title
            safe_title = (a["title"]
                          .replace("&", "&amp;")
                          .replace("<", "&lt;")
                          .replace(">", "&gt;")
                          .replace('"', "&quot;"))
            rows_html += (
                f'      <li>\n'
                f'        <a href="{a["url"]}" target="_blank" rel="noopener">'
                f'{safe_title}</a>\n'
                f'      </li>'
            )
        rows_html += "\n    </ol>\n    </div>"

    summary_block = ""
    if ai_summary and "विफल" not in ai_summary:
        safe_sum = (ai_summary
                    .replace("&", "&amp;")
                    .replace("<", "&lt;")
                    .replace(">", "&gt;"))
        summary_block = f"""
    <div class="ai-summary">
      <h2>🤖 AI दैनिक सारांश  <span class="badge">{OLLAMA_MODEL}</span></h2>
      <p>{safe_sum}</p>
    </div>"""

    html = f"""<!DOCTYPE html>
<html lang="ne">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>नेपाल समाचार — {today}</title>
  <style>
    @import url('https://fonts.googleapis.com/css2?family=Mukta:wght@400;600;700&display=swap');
    :root {{
      --bg      : #0e1117;
      --card    : #181c25;
      --accent  : #d4393e;
      --text    : #dce0e8;
      --link    : #5eb1ef;
      --link-hv : #90caf9;
      --muted   : #7a7f8a;
      --border  : #252a36;
    }}
    * {{ box-sizing:border-box; margin:0; padding:0; }}
    body {{
      font-family:'Mukta','Noto Sans Devanagari',sans-serif;
      background:var(--bg); color:var(--text);
      line-height:1.7; padding:2rem 1rem;
    }}
    .container {{ max-width:840px; margin:0 auto; }}
    header {{
      text-align:center; margin-bottom:2rem;
      padding-bottom:1.4rem; border-bottom:2px solid var(--accent);
    }}
    header h1 {{ font-size:2rem; color:#fff; }}
    header p  {{ color:var(--muted); margin-top:.35rem; font-size:.92rem; }}
    .ai-summary {{
      background:linear-gradient(135deg,#162030,#1a2640);
      border-left:4px solid var(--link);
      border-radius:8px; padding:1.2rem 1.5rem; margin-bottom:1.8rem;
    }}
    .ai-summary h2 {{ font-size:1.05rem; color:var(--link); margin-bottom:.5rem; }}
    .ai-summary p  {{ font-size:.95rem; color:#cdd3de; line-height:1.8; }}
    .source-block {{
      background:var(--card); border-radius:10px;
      padding:1.1rem 1.4rem; margin-bottom:1.1rem;
      border:1px solid var(--border);
    }}
    .source-block h2 {{ font-size:1rem; color:var(--accent); margin-bottom:.6rem; }}
    ol {{ padding-left:1.4rem; }}
    li {{ padding:.32rem 0; border-bottom:1px solid var(--border); }}
    li:last-child {{ border-bottom:none; }}
    a {{ color:var(--link); text-decoration:none; }}
    a:hover {{ color:var(--link-hv); text-decoration:underline; }}
    .badge {{
      display:inline-block; background:var(--accent); color:#fff;
      font-size:.68rem; padding:.1rem .5rem; border-radius:10px;
      margin-left:.4rem; vertical-align:middle;
    }}
    footer {{ text-align:center; margin-top:2rem; color:var(--muted); font-size:.78rem; }}
  </style>
</head>
<body>
  <div class="container">
    <header>
      <h1>🇳🇵 नेपाल समाचार — {today}</h1>
      <p>शीर्ष {len(articles)} समाचार &middot; {len(grouped)} स्रोत &middot; Python + Ollama द्वारा तयार पारिएको</p>
    </header>
    {summary_block}
    {rows_html}
    <footer>
      {datetime.now().strftime("%H:%M:%S")} मा एकत्र गरिएको
      &middot; Ollama ({OLLAMA_MODEL})
    </footer>
  </div>
</body>
</html>"""

    with open(output, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"\n✅ HTML सेभ भयो → {os.path.abspath(output)}")
    return output


html_path = build_html(top_articles, ai_summary=ai_summary)




✅ HTML सेभ भयो → C:\Users\kusha\AgenticAIWs\NepalNewsAgent\nepali_news.html


In [7]:
#Inline preview + JSON
try:
    from IPython.display import HTML, display
    with open(html_path, encoding="utf-8") as f:
        display(HTML(f.read()))
except Exception:
    print(f"ब्राउजरमा खोल्नुहोस्: file://{html_path}")

json_path = str(OUTPUT_DIR / "nepali_news.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump({
        "date":       datetime.now().isoformat(),
        "model":      OLLAMA_MODEL,
        "ai_summary": ai_summary,
        "articles":   top_articles,
    }, f, ensure_ascii=False, indent=2)
print(f"✅ JSON → {os.path.abspath(json_path)}")




✅ JSON → C:\Users\kusha\AgenticAIWs\NepalNewsAgent\nepali_news.json
